In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [2]:
def conv_block(inputs, num_filters):
    # Two 3x3 Convolutions with ReLU activation
    x = layers.Conv2D(num_filters, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(num_filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x

In [3]:
def build_unet(input_shape=(128, 128, 1)):
    inputs = layers.Input(input_shape)

    # --- Encoder (Downsampling) ---
    s1 = conv_block(inputs, 64)
    p1 = layers.MaxPooling2D((2, 2))(s1)

    s2 = conv_block(p1, 128)
    p2 = layers.MaxPooling2D((2, 2))(s2)

    # --- Bottleneck ---
    b1 = conv_block(p2, 256)

    # --- Decoder (Upsampling) ---
    u1 = layers.Conv2DTranspose(128, (2, 2), strides=2, padding="same")(b1)
    u1 = layers.concatenate([u1, s2]) # Skip Connection!
    u1 = conv_block(u1, 128)

    u2 = layers.Conv2DTranspose(64, (2, 2), strides=2, padding="same")(u1)
    u2 = layers.concatenate([u2, s1]) # Skip Connection!
    u2 = conv_block(u2, 64)

    # --- Output Layer ---
    outputs = layers.Conv2D(1, 1, padding="same", activation="sigmoid")(u2)

    return models.Model(inputs, outputs, name="U-Net_Brain_Tumor")

In [4]:
model = build_unet()
model.summary()

Model: "U-Net_Brain_Tumor"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 128,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │     36,928 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │    147,584 │ activation_2[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ activation_3[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │      1,024 │ conv2d_4[0][0]  

 Total params: 1,866,817 (7.12 MB)

 Trainable params: 1,864,257 (7.11 MB)

 Non-trainable params: 2,560 (10.00 KB)

In [5]:
def dice_coef(y_true, y_pred):
    smooth = 1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

In [6]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.utils import Sequence

In [ ]:
class BrainDataGenerator(Sequence):
    def __init__(self, image_dir, mask_dir, file_list=None, batch_size=16, img_size=128):
        self.image_dir   = image_dir
        self.mask_dir    = mask_dir
        self.batch_size  = batch_size
        self.img_size    = img_size
        # Use given file list OR load everything
        self.image_files = file_list if file_list else sorted(os.listdir(image_dir))

    def __len__(self):
        return len(self.image_files) // self.batch_size

    def __getitem__(self, idx):
        batch_files = self.image_files[idx * self.batch_size : (idx + 1) * self.batch_size]
        X = np.empty((self.batch_size, self.img_size, self.img_size, 1))
        y = np.empty((self.batch_size, self.img_size, self.img_size, 1))

        for i, file_name in enumerate(batch_files):
            img  = np.load(os.path.join(self.image_dir, file_name))
            mask = np.load(os.path.join(self.mask_dir,  file_name))
            X[i, :, :, 0] = img
            y[i, :, :, 0] = mask

        return X, y


In [13]:
from tensorflow.keras.optimizers import Adam

# Build the model we coded in Step 2
model = build_unet()

# Compile with our custom Dice Loss and Dice Coefficient
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=dice_loss, 
    metrics=[dice_coef, 'accuracy']
)



In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/brain_tumor_unet.h5',  # ← saved to Drive!
    monitor='val_dice_coef',
    verbose=1,
    save_best_only=True,
    mode='max'
)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)

callbacks = [checkpoint, early_stop, reduce_lr]


In [ ]:
import random

# --- Paths (Colab + unzipped data) ---
IMAGE_DIR = '/content/Brain_Data_Clean/images'
MASK_DIR  = '/content/Brain_Data_Clean/masks'
BATCH_SIZE = 16
EPOCHS     = 50

# --- 80/20 Train/Val Split ---
all_files = sorted(os.listdir(IMAGE_DIR))
random.seed(42)
random.shuffle(all_files)

split_idx   = int(0.8 * len(all_files))
train_files = all_files[:split_idx]
val_files   = all_files[split_idx:]

print(f"✅ Train: {len(train_files)} | Val: {len(val_files)}")

# --- Generators ---
train_gen = BrainDataGenerator(IMAGE_DIR, MASK_DIR, file_list=train_files, batch_size=BATCH_SIZE)
val_gen   = BrainDataGenerator(IMAGE_DIR, MASK_DIR, file_list=val_files,   batch_size=BATCH_SIZE)

# --- Train! ---
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


FileNotFoundError: [Errno 2] No such file or directory: 'Brain_Data_Clean/images'

In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


[]


In [2]:
import tensorflow as tf
print(tf.__version__)


2.20.0
